# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page for one client on one reporting day. I verified this using report_date, client_hash_id, and content_hash_id; there were 0 duplicate combinations, confirming the page-day grain.

**Time window:** For this assignment, I will use the March 2026 partition (2026-03-01 to 2026-03-31) as my development and verification window. I chose a mid-panel month so I can develop and test the data contract and features without using the final June 2026 month, which should be treated as a sealed/future test period.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", bool(HF_TOKEN))

Token loaded: True


In [3]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset:", info.id)
print("Access confirmed.")

Dataset: FlyRank/internship-warehouse
Access confirmed.


In [4]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="README.md",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Warehouse connection working.")
print(path)

Warehouse connection working.
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/README.md


In [5]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [6]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_path)

print("Rows:", len(march_df))
print("Columns:", len(march_df.columns))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

Rows: 9841378
Columns: 30
Date range: 2026-03-01 to 2026-03-31


In [7]:
print(march_df.columns.tolist())
march_df.head()

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
duplicates = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate page-day combinations:", len(duplicates))

Duplicate page-day combinations: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# Profile all 30 columns
column_profile = pd.DataFrame({
    "column": march_df.columns,
    "non_null_count": march_df.notna().sum().values,
    "missing_count": march_df.isna().sum().values,
    "non_null_pct": (march_df.notna().mean() * 100).round(2).values
})

column_profile = column_profile.sort_values(
    "non_null_pct",
    ascending=False
)

column_profile

,column,non_null_count,missing_count,non_null_pct
0,report_date,9841378,0,100.00
1,client_hash_id,9841378,0,100.00
2,content_hash_id,9841378,0,100.00
3,client_has_gsc,9841378,0,100.00
4,client_has_ga4,9841378,0,100.00
5,gsc_data_available,9841378,0,100.00
7,gsc_impressions,9841378,0,100.00
8,gsc_clicks,9841378,0,100.00
9,gsc_sum_position,9841378,0,100.00
6,ga4_data_available,6822637,3018741,69.33


In [10]:
# Check whether metric values are actually present when the data source
# is marked unavailable

print("GSC availability vs impressions:")
print(pd.crosstab(
    march_df["gsc_data_available"],
    march_df["gsc_impressions"].notna()
))

print("\nGSC availability vs average position:")
print(pd.crosstab(
    march_df["gsc_data_available"],
    march_df["gsc_avg_position"].notna()
))

print("\nGA4 availability vs pageviews:")
print(pd.crosstab(
    march_df["ga4_data_available"],
    march_df["ga4_pageviews"].notna()
))

GSC availability vs impressions:
gsc_impressions        True
gsc_data_available         
False               6230317
True                3611061

GSC availability vs average position:
gsc_avg_position      False    True 
gsc_data_available                  
False               6230317        0
True                      0  3611061

GA4 availability vs pageviews:
ga4_pageviews          True
ga4_data_available         
False               6408671
True                 413966


In [11]:
ga4_check = march_df.groupby("ga4_data_available").agg(
    rows=("content_hash_id", "size"),
    pageviews_non_null=("ga4_pageviews", "count"),
    sessions_non_null=("ga4_sessions", "count"),
    engaged_sessions_non_null=("ga4_engaged_sessions", "count"),
    engagement_sec_non_null=("ga4_total_engagement_sec", "count")
)

ga4_check

,rows,pageviews_non_null,sessions_non_null,engaged_sessions_non_null,engagement_sec_non_null
ga4_data_available,,,,,
False,6408671,6408671,6408671,6408671,6408671
True,413966,413966,413966,413966,413966


In [12]:
march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].describe()

march_df[
    [
        "sessions_organic",
        "sessions_ai"
    ]
].isna().sum()

,0
sessions_organic,3018741
sessions_ai,3018741


**2. Fields: Feature / Label / Context / Excluded**

**Context**

These fields help identify, organize, filter, or interpret the data, but they will not be used by the model as predictive features.

**report_date **— identifies the date of the observation.
**client_hash_id** — identifies the client and can be used for grouping or splitting.
**content_hash_id** — identifies the page/content item.
**client_has_gsc** — indicates whether the client has GSC.
**client_has_ga4** — indicates whether the client has GA4.
**gsc_data_available** — indicates whether usable GSC data is available for the observation.
**ga4_data_available** — indicates whether usable GA4 data is available for the observation.

**Features**

**For this assignment, I will use the following five initial features:**

**gsc_impressions **— measures the page's search visibility.
**gsc_clicks** — measures the search traffic received by the page.
**gsc_avg_position** — represents the page's average search position.
**sessions_organic** — measures organic sessions reaching the page.
**sessions_ai** — measures sessions attributed to AI sources.

Together, these features provide information about search visibility, search traffic, search ranking, organic traffic, and AI-driven traffic.

The availability of these features will also be considered during preprocessing. In particular, gsc_avg_position is available for 36.69% of rows, while sessions_organic and sessions_ai are available for 69.33% of rows, so missing values will not automatically be treated as zero.

**Label / Proxy**

There is no direct label in the March performance fields that tells us whether a page should receive priority.

Therefore, I will not use a current performance metric as the label. Instead, the label/proxy will be derived from an observed future outcome window, allowing the model to use information available earlier to assess what happens to the page later.

**Excluded**
gsc_sum_position — excluded because it is closely related to gsc_avg_position and provides redundant position information.
ga4_pageviews — excluded because it overlaps with other traffic measures and is less directly aligned with our selected feature set.
ga4_sessions — excluded because it overlaps with other traffic-volume measures.
ga4_users — excluded because it provides another closely related measure of traffic volume.
ga4_engaged_sessions — excluded from the final five because we selected sessions_organic and sessions_ai as more directly relevant traffic signals for this initial frame.
ga4_total_engagement_sec — excluded for the same reason; it is useful engagement information but is not part of our selected five-feature frame.
sessions_direct — excluded because direct traffic is less directly aligned with our organic content-prioritization decision.
sessions_referral — excluded because referral traffic is less directly aligned with the current decision.
sessions_social — excluded because social traffic is less directly aligned with the current decision.
sessions_paid — excluded because paid traffic is less directly aligned with the organic content-prioritization decision.
ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — excluded because they are detailed components of AI traffic and would add unnecessary granularity and redundancy when sessions_ai already captures the broader AI-traffic signal.
scroll_events — excluded because it is a relatively crude engagement signal and is less directly useful than the selected traffic and search-performance features.

Excluded does not mean these fields are useless. It means that for this initial five-feature frame, we deliberately chose not to use them because they are redundant, less aligned with the decision, or less informative than the selected features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
total_rows = len(march_df)

gsc_available = march_df["gsc_data_available"].eq(True).sum()
ga4_available = march_df["ga4_data_available"].eq(True).sum()

print("Total March rows:", total_rows)
print("GSC available rows:", gsc_available)
print("GA4 available rows:", ga4_available)

print("GSC available %:", round(gsc_available / total_rows * 100, 2))
print("GA4 available %:", round(ga4_available / total_rows * 100, 2))

Total March rows: 9841378
GSC available rows: 3611061
GA4 available rows: 413966
GSC available %: 36.69
GA4 available %: 4.21


**3.1 Grain**

I verified that the combination of report_date, client_hash_id, and content_hash_id contains no duplicate records in the March 2026 data.

The result was 0 duplicate page-day combinations, confirming that one row represents one page for one client on one reporting date.

**3.2 Row Count and Date Window**

The March 2026 slice contains 9,841,378 rows, with a date range from March 1, 2026 to March 31, 2026.

Therefore, our analysis is based on the March 2026 data window.

**3.3 Data Availability**

Out of the 9,841,378 March rows:

**GSC data:** 3,611,061 rows (36.69%) available
**GA4 data:** 413,966 rows (4.21%) available



**3.4 Perform the leakage trap**

In [14]:
features_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].copy()

features_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,sessions_ai
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


In [15]:
features_df.shape
features_df.isna().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,6230317
sessions_organic,3018741
sessions_ai,3018741


In [16]:
from datasets import load_dataset

april = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-04/data_0.parquet",
    split="train"
)

april_df = april.to_pandas()

print(april_df.shape)
print(april_df.columns.tolist())

(10424730, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [19]:
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March path:", march_path)
print("April path:", april_path)

March path: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April path: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet


In [20]:
import duckdb

con = duckdb.connect()

result = con.execute(f"""
    SELECT
        COUNT(*) AS matched_rows,
        COUNT(*) FILTER (
            WHERE a.gsc_impressions IS NOT NULL
        ) AS april_impressions_available,
        COUNT(*) FILTER (
            WHERE a.gsc_clicks IS NOT NULL
        ) AS april_clicks_available
    FROM read_parquet('{march_path}') m
    INNER JOIN read_parquet('{april_path}') a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id
""").fetchone()

print("Matched rows:", result[0])
print("April impressions available:", result[1])
print("April clicks available:", result[2])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Matched rows: 295241310
April impressions available: 295241310
April clicks available: 295241310


In [21]:
march_april_summary = con.execute(f"""
WITH march_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(sessions_organic) AS march_sessions_organic,
        SUM(sessions_ai) AS march_sessions_ai
    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

april_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,
        AVG(gsc_avg_position) AS april_avg_position,
        SUM(sessions_organic) AS april_sessions_organic,
        SUM(sessions_ai) AS april_sessions_ai
    FROM read_parquet('{april_path}')
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.march_impressions,
    a.april_impressions,

    m.march_clicks,
    a.april_clicks,

    m.march_avg_position,
    a.april_avg_position,

    m.march_sessions_organic,
    a.april_sessions_organic,

    m.march_sessions_ai,
    a.april_sessions_ai

FROM march_month m
INNER JOIN april_month a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
""").fetchdf()

print("Matched page-client pairs:", len(march_april_summary))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Matched page-client pairs: 331436


In [22]:
march_april_summary["impressions_change"] = (
    march_april_summary["april_impressions"]
    - march_april_summary["march_impressions"]
)

march_april_summary["clicks_change"] = (
    march_april_summary["april_clicks"]
    - march_april_summary["march_clicks"]
)

march_april_summary["organic_change"] = (
    march_april_summary["april_sessions_organic"]
    - march_april_summary["march_sessions_organic"]
)

march_april_summary["ai_change"] = (
    march_april_summary["april_sessions_ai"]
    - march_april_summary["march_sessions_ai"]
)

march_april_summary[
    [
        "impressions_change",
        "clicks_change",
        "organic_change",
        "ai_change"
    ]
].describe()

,impressions_change,clicks_change,organic_change,ai_change
count,331436.000000,331436.000000,260736.000000,260736.000000
mean,16.133812,-0.047451,1.045897,0.011176
std,2181.767992,9.401169,15.102345,0.564662
min,-162069.000000,-912.000000,-1162.000000,-45.000000
25%,-11.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,261817.000000,1766.000000,1948.000000,53.000000


In [23]:
import numpy as np

march_april_summary["impressions_pct_change"] = np.where(
    march_april_summary["march_impressions"] > 0,
    (march_april_summary["april_impressions"]
     - march_april_summary["march_impressions"])
    / march_april_summary["march_impressions"] * 100,
    np.nan
)

march_april_summary["clicks_pct_change"] = np.where(
    march_april_summary["march_clicks"] > 0,
    (march_april_summary["april_clicks"]
     - march_april_summary["march_clicks"])
    / march_april_summary["march_clicks"] * 100,
    np.nan
)

march_april_summary["organic_pct_change"] = np.where(
    march_april_summary["march_sessions_organic"] > 0,
    (march_april_summary["april_sessions_organic"]
     - march_april_summary["march_sessions_organic"])
    / march_april_summary["march_sessions_organic"] * 100,
    np.nan
)

march_april_summary[
    [
        "impressions_pct_change",
        "clicks_pct_change",
        "organic_pct_change"
    ]
].describe()

,impressions_pct_change,clicks_pct_change,organic_pct_change
count,176737.000000,68837.000000,43187.000000
mean,165.019554,-9.682276,68.009025
std,3780.737169,234.956440,329.021513
min,-100.000000,-100.000000,-100.000000
25%,-63.600000,-100.000000,-100.000000
50%,-25.107604,-50.000000,-14.285714
75%,30.000000,0.000000,100.000000
max,604800.000000,28950.000000,11400.000000


In [24]:
print("March impressions = 0:",
      (march_april_summary["march_impressions"] == 0).sum())

print("March impressions < 10:",
      (march_april_summary["march_impressions"] < 10).sum())

print("March clicks = 0:",
      (march_april_summary["march_clicks"] == 0).sum())

print("March clicks < 5:",
      (march_april_summary["march_clicks"] < 5).sum())

March impressions = 0: 154699
March impressions < 10: 188230
March clicks = 0: 262599
March clicks < 5: 302618


In [25]:
march_april_summary["march_impressions"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

,march_impressions
count,331436.000000
mean,846.792708
std,4044.520587
min,0.000000
10%,0.000000
25%,0.000000
50%,2.000000
75%,216.000000
90%,1707.000000
95%,4225.000000


In [26]:
thresholds = [
    (50, 20),
    (50, 30),
    (50, 50),
    (100, 20),
    (100, 30),
    (100, 50),
    (200, 20),
    (200, 30),
    (200, 50),
]

for baseline, decline in thresholds:
    eligible = march_april_summary[
        march_april_summary["march_impressions"] >= baseline
    ]

    labeled = eligible[
        eligible["april_impressions"]
        <= eligible["march_impressions"] * (1 - decline / 100)
    ]

    print(
        f"Baseline >= {baseline}, "
        f"Decline >= {decline}%: "
        f"{len(labeled):,} labels / {len(eligible):,} eligible "
        f"({len(labeled) / len(eligible) * 100:.2f}%)"
    )

Baseline >= 50, Decline >= 20%: 60,261 labels / 116,114 eligible (51.90%)
Baseline >= 50, Decline >= 30%: 50,740 labels / 116,114 eligible (43.70%)
Baseline >= 50, Decline >= 50%: 31,058 labels / 116,114 eligible (26.75%)
Baseline >= 100, Decline >= 20%: 52,533 labels / 101,441 eligible (51.79%)
Baseline >= 100, Decline >= 30%: 43,887 labels / 101,441 eligible (43.26%)
Baseline >= 100, Decline >= 50%: 26,125 labels / 101,441 eligible (25.75%)
Baseline >= 200, Decline >= 20%: 43,532 labels / 84,833 eligible (51.31%)
Baseline >= 200, Decline >= 30%: 35,951 labels / 84,833 eligible (42.38%)
Baseline >= 200, Decline >= 50%: 20,582 labels / 84,833 eligible (24.26%)


**Label / Proxy Definition**

The dataset does not contain a direct column indicating whether a page should be prioritized. Therefore, the label is derived from future observed performance rather than taken directly from an existing column.

The March 2026 data represents the information available at the time of the decision, while April 2026 is used as the future outcome window.

To avoid unstable percentage changes from very low-volume pages, only page-client pairs with at least 100 March GSC impressions are included in the eligible modeling population.

A page receives a positive label (1) if its April GSC impressions are at least 30% lower than its March GSC impressions:

March impressions ≥ 100 AND April impressions ≤ 70% of March impressions

Otherwise, the label is 0.

This produces 101,441 eligible page-client pairs, with approximately 43.26% positive labels and 56.74% negative labels.

The label is therefore a new derived variable, not an existing dataset column. April performance is used only to construct the label and will not be used as a model feature, preventing future-data leakage.

**In simple terms:** the model will learn from what a page looked like in March and predict whether that page will experience a meaningful search-visibility decline in April.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**This dataset has several limitations that affect what the model can reliably tell us.**


**Uneven data availability:** GSC and GA4 data are not available for every page and date. In March 2026, GSC data is available for 36.69% of rows, while GA4 data availability is only 4.21% according to the availability flags. Therefore, missing data cannot automatically be interpreted as zero activity.

**GSC-only history:** Some earlier observations may contain GSC information without corresponding GA4 information. This means we cannot assume that every page has the same history or the same set of measurable signals.

**Different observation windows:** The available fields may represent different reporting or collection windows. We therefore cannot assume that every column describes exactly the same period without checking its definition.

**Future outcomes are not directly observed in the current row:** The March performance data tells us what was known at that time, but it does not directly tell us whether a page will decline or require attention later. A future outcome must be derived from a later observation window.

**No causal explanation:**
The data can show patterns and associations between page characteristics and later outcomes, but it cannot prove that one feature directly caused a page's performance to change.

**Limited generalization:** The model will learn from the historical pages and clients represented in this warehouse. Its predictions may not perform equally well for new clients, pages, or situations that are very different from the historical data.

**Overall limitation:** This dataset can help us identify and rank pages based on observed patterns, but it cannot guarantee why a page changed, what caused the change, or what will happen in every future situation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.